# Task 3: Event Impact Modeling

Translate `impact_link` relationships into an event–indicator association matrix, validate against historical change, and refine magnitudes.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_loader import load_unified_data, load_impact_sheet
from src.impact_model import (
    join_events_impacts, impact_summary, build_association_matrix,
    cumulative_event_effects, validate_telebirr_mm_impact
)

FIG = ROOT / 'reports' / 'figures'
MODEL = ROOT / 'models'
FIG.mkdir(parents=True, exist_ok=True)
MODEL.mkdir(exist_ok=True)

data = load_unified_data()
impacts = load_impact_sheet()
joined = join_events_impacts(data, impacts)
summary = impact_summary(joined)
print(summary)
summary.to_csv(MODEL/'event_impact_summary.csv', index=False)


In [ ]:
matrix = build_association_matrix(joined)
print(matrix)
matrix.to_csv(MODEL/'association_matrix.csv')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Event-Indicator Association Matrix (estimated effect)')
fig.tight_layout()
fig.savefig(FIG/'association_matrix.png', dpi=150)
plt.show()


## Functional form
- Each impact_link contributes an additive percentage-point shock.
- Effects apply fully after `lag_months` (`step_after_lag`), or ramp linearly (`linear_ramp`).
- Multiple events stack additively; for Findex Access we later down-scale because operator-era shocks overstate survey ownership.


In [ ]:
years = list(range(2011, 2028))
access_path, access_contrib = cumulative_event_effects(joined, 'ACC_OWNERSHIP', years, form='step_after_lag')
usage_path, usage_contrib = cumulative_event_effects(joined, 'USG_DIGITAL_PAYMENT', years, form='linear_ramp')
print(access_path.tail(10))
print(usage_path.tail(10))
access_path.to_csv(MODEL/'access_event_path.csv', index=False)
usage_path.to_csv(MODEL/'usage_event_path.csv', index=False)

validation = validate_telebirr_mm_impact(joined)
print(validation)


## Validation notes
- Observed MM survey ownership: **4.7% -> 9.45% (+4.75pp)** after Telebirr era.
- Observed Access ownership: **46% -> 49% (+3pp)** 2021-2024.
- Raw sum of Access impact links exceeds observed change -> refined IMP_0001 to ~4pp and apply scale factor in forecasting.
- Comparable-country (Kenya) evidence informs Usage boosts but must be tempered for Ethiopia's banked-but-dormant dynamics (Market Nuances).

## Assumptions & uncertainties
1. Impact magnitudes are expert/compiled estimates, not causal identification.
2. Linear additive stacking ignores interaction/saturation.
3. Lags are approximate.
4. Operator metrics != Findex definitions.
